# S4 · Calidad de la medición: error e incertidumbre · ¿Cuánto puedo confiar en mi registrador?
**Sistemas de adquisición de datos · TSU en Electrónica y Pruebas · Jabil**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggt13/DAQ_TSU/blob/main/sesion_04_error_e_incertidumbre/S4_AC_CLASE_Colab_ALUMNO.ipynb)

## Objetivo
Expresar una medición como **valor ± incertidumbre** y decidir si un registrador es **compatible** con una
referencia: separar lo que aporta el ruido (tipo A) de lo que aporta el fabricante (tipo B) y combinarlos.

## La situación
Antes de confiar en un registrador para una auditoría hay que saber cuánto puede equivocarse. En clase trabajas con
las lecturas de **tu** registrador junto a un termómetro de referencia; en la práctica, con los tres Elitech RC-5 del
grupo puestos uno junto al otro, cada alumno con un registrador y una hora distintos.

## En qué consiste este cuaderno
| Parte | Tiempo | Qué haces | Qué produce |
|---|---|---|---|
| **A · Actividad en clase** | 30 min | Cuatro ejercicios (★ primero): incertidumbre tipo A, tipo B, combinada y expandida. En cada uno escribes la expresión, la sustitución con tus números y su unidad, y lo que predice. Cierre: ¿tu registrador es compatible? | Tu resultado como valor ± U y tu veredicto |
| **B · Práctica** | 80 min | Tomas una hora de **tu** RC-5 en la intercomparación del grupo, la comparas contra la mediana de los otros dos y decides si es compatible | Gráficas firmadas, tu conclusión y el archivo `S4_<código>.csv` |
| **Hoja de resultados** | en la práctica | Importas el CSV en tu hoja de cálculo de Google y contestas las preguntas | El reporte que entregas |

## Qué entregas y cómo se califica
1. **Este cuaderno**, al terminar la sesión, con todo ejecutado. ★ 3 pts · ejercicios 2, 3 y 4, 2 pts cada uno ·
   cierre 1 pt. Se califica que el resultado sea correcto para **tus** datos y que la sustitución y la predicción
   sean tuyas.
2. **La hoja de resultados** con el reporte, en 7 días naturales.

## Reglas
- **Archivo → Guardar una copia en Drive** antes de empezar; trabaja en tu copia.
- Completa sólo lo marcado con ✎. Ninguna celda avanza si falta algo: el mensaje dice qué.
- La revisión automática dice si el ejercicio está **completo**, no si está **bien**; eso lo califica el profesor.
- Si usas un asistente de IA, dilo en tu reporte. Las sustituciones y las respuestas llevan **tus** números.

In [ ]:
#@title Herramientas del curso (no modificar)
"""Herramientas comunes del curso «Sistemas de adquisición de datos» 2027.

Identificación del alumno, datos personales a partir de su código, firma de las gráficas,
huella de los resultados y CSV firmado para la hoja de resultados.

Este archivo es la fuente única: los cuadernos de cada sesión copian su contenido en la celda
«Herramientas del curso». Si se corrige aquí, se regeneran los cuadernos.
"""
import csv
import datetime as _dt
import hashlib
import zlib

import numpy as np

try:
    from zoneinfo import ZoneInfo
    _ZONA = ZoneInfo("America/Mexico_City")
except Exception:          # Python sin zoneinfo: se usa la hora del sistema
    _ZONA = None

_VALORES_DE_EJEMPLO = {"", "TU NOMBRE", "NOMBRE APELLIDO", "000000000", "123456789"}


class Alumno:
    """Datos de quien ejecuta el cuaderno. Se crea con identificar()."""

    def __init__(self, nombre, codigo, sesion):
        self.nombre = nombre
        self.codigo = codigo
        self.sesion = sesion

    @property
    def semilla(self):
        return int(self.codigo) % (2 ** 32)

    def rng(self, etiqueta=""):
        """Generador de números aleatorios propio del alumno.

        Cada etiqueta da una secuencia independiente: agregar un parámetro nuevo
        no cambia los que ya existían.
        """
        return np.random.default_rng([self.semilla, zlib.crc32(etiqueta.encode("utf-8"))])

    def __repr__(self):
        return f"Alumno({self.nombre!r}, {self.codigo!r}, {self.sesion!r})"


def identificar(nombre, codigo, sesion):
    """Valida la celda de identificación y devuelve el Alumno.

    Detiene el cuaderno con un mensaje claro si el nombre o el código faltan
    o se dejaron con el valor de ejemplo.
    """
    nombre = " ".join(str(nombre).split())
    codigo = str(codigo).strip()
    if nombre.upper() in _VALORES_DE_EJEMPLO or len(nombre.split()) < 2:
        raise ValueError("Escribe tu nombre completo (nombre y apellido) en NOMBRE y vuelve a ejecutar.")
    if codigo in _VALORES_DE_EJEMPLO or not codigo.isdigit() or not 7 <= len(codigo) <= 10:
        raise ValueError("Escribe tu código de alumno (sólo dígitos, de 7 a 10) en CODIGO y vuelve a ejecutar.")
    return Alumno(nombre, codigo, sesion)


def parametro(alumno, etiqueta, minimo, maximo, decimales=3):
    """Un valor personal entre minimo y maximo, siempre el mismo para el mismo código."""
    valor = alumno.rng(etiqueta).uniform(minimo, maximo)
    return round(float(valor), decimales)


def asignar(alumno, opciones, salto=0):
    """Elige una opción de la lista según el código (estación, ventana de fechas, carga del motor...)."""
    opciones = list(opciones)
    return opciones[(int(alumno.codigo) + salto) % len(opciones)]


def _redondea(valor):
    if isinstance(valor, (int, float, np.floating, np.integer)):
        return format(float(valor), ".4g")
    return str(valor)


def huella(alumno, resultados):
    """Seis caracteres que dependen del código, la sesión y los resultados (redondeados a 4 cifras)."""
    partes = [alumno.codigo, alumno.sesion] + [f"{k}={_redondea(v)}" for k, v in sorted(resultados.items())]
    return hashlib.sha256("|".join(partes).encode("utf-8")).hexdigest()[:6].upper()


def ahora():
    momento = _dt.datetime.now(_ZONA) if _ZONA else _dt.datetime.now()
    return momento.strftime("%Y-%m-%d %H:%M")


def titulo(ax, texto, alumno):
    """Título de la gráfica con el nombre del alumno."""
    ax.set_title(f"{texto} · {alumno.nombre}")


def firma(fig, alumno, resultados=None):
    """Pie de la figura: nombre · código · sesión · fecha y hora · huella.

    Toda gráfica que se entrega debe pasar por aquí.
    """
    texto = f"{alumno.nombre} · {alumno.codigo} · {alumno.sesion} · {ahora()}"
    if resultados:
        texto += f" · huella {huella(alumno, resultados)}"
    fig.text(0.01, 0.005, texto, fontsize=8, color="0.35", ha="left", va="bottom")
    fig.subplots_adjust(bottom=max(fig.subplotpars.bottom, 0.14))
    fig._firmada = True
    return fig


def exportar_csv(ruta, alumno, resultados, columnas, extra=None):
    """Escribe el CSV firmado que se importa en la pestaña «CSV» de la hoja de resultados.

    Renglones de metadatos: nombre, codigo, sesion, huella, ejecutado y los de `extra` (p. ej. la fuente
    de los datos). Después, un renglón por resultado («resultado:<nombre>», valor). Luego un renglón vacío
    y la tabla: encabezados y datos. `columnas` es un diccionario {encabezado: lista de valores}, todas de
    la misma longitud. La hoja busca cada dato por su etiqueta, no por su renglón.
    """
    claves = [k.lower() for k in resultados]
    if len(set(claves)) != len(claves):    # la hoja busca sin distinguir mayúsculas: U y u serían el mismo dato
        raise ValueError("Dos resultados tienen el mismo nombre salvo mayúsculas; renómbrelos.")
    largos = {len(v) for v in columnas.values()}
    if len(largos) != 1:
        raise ValueError("Todas las columnas de la tabla deben tener el mismo número de renglones.")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["nombre", alumno.nombre])
        w.writerow(["codigo", alumno.codigo])
        w.writerow(["sesion", alumno.sesion])
        w.writerow(["huella", huella(alumno, resultados)])
        w.writerow(["ejecutado", ahora()])
        for k, v in (extra or {}).items():
            w.writerow([k, v])
        for k, v in resultados.items():
            w.writerow([f"resultado:{k}", _redondea(v)])
        w.writerow([])
        w.writerow(list(columnas))
        for fila in zip(*columnas.values()):
            w.writerow([_redondea(x) for x in fila])
    return ruta


def leer_csv_firmado(ruta):
    """Para el profesor: devuelve (metadatos, resultados, tabla) de un CSV firmado."""
    meta, resultados, tabla = {}, {}, []
    with open(ruta, newline="", encoding="utf-8") as f:
        filas = list(csv.reader(f))
    i = 0
    while i < len(filas) and filas[i]:
        clave, valor = filas[i][0], filas[i][1]
        if clave.startswith("resultado:"):
            resultados[clave.split(":", 1)[1]] = float(valor)
        else:
            meta[clave] = valor
        i += 1
    tabla = filas[i + 1:] if i + 1 < len(filas) else []
    return meta, resultados, tabla


def verificar_csv(ruta):
    """Para el profesor: True si la huella del CSV corresponde a su código, sesión y resultados."""
    meta, resultados, _ = leer_csv_firmado(ruta)
    alumno = Alumno(meta["nombre"], meta["codigo"], meta["sesion"])
    return huella(alumno, resultados) == meta["huella"]


# ---------------------------------------------------------------------------------------------
# Revisión de lo que escribe el alumno. Comprueba que cada ejercicio esté COMPLETO, no que esté
# bien: el resultado correcto lo califica el profesor con la clave. Si falta algo, detiene el
# cuaderno y dice qué falta.
# ---------------------------------------------------------------------------------------------
import math
import re

REGISTRO = {}


def _normaliza(texto):
    t = str(texto).replace("\u202f", "").replace("\u00a0", "").replace(" ", "")
    t = re.sub(r"(?<![\d.])([1-9]\d{0,2})((?:,\d{3})+)(?![\d])",  # separador de miles: 32,000 -> 32000
               lambda m: m.group(1) + m.group(2).replace(",", ""), t)
    return t.replace(",", ".")                                       # coma decimal: 0,068 -> 0.068


def _formas(x):
    """Maneras razonables de escribir un número: entero, o redondeado a 2, 3 o 4 cifras significativas."""
    x = float(x)
    formas = set()
    if x == int(x) and abs(x) < 1e9:
        formas.add(str(int(x)))
    if abs(x) >= 100:
        formas.add(str(int(round(x))))                       # 34271.6 escrito como 34272
    if abs(x) >= 1:
        for dec in (1, 2):
            formas.add(f"{x:.{dec}f}".rstrip("0").rstrip("."))  # 123456.7 con sus decimales
    for cifras in (2, 3, 4, 5, 6):
        g = f"{x:.{cifras}g}"
        if "e" not in g:
            formas.add(g)
    return {f for f in formas if f not in ("", "-0")}


def aparece(x, texto):
    """True si el número x está escrito en el texto (sin confundir 0.03 con 0.0337)."""
    t = _normaliza(texto)
    def patron(f):
        cola = r"0*" if "." in f else r"(?:\.0*)?"            # 0.0342 = 0.034200 · 120 = 120.0, pero no 1200
        return r"(?<![\d.])" + re.escape(f) + cola + r"(?![\d])"
    return any(re.search(patron(f), t) for f in _formas(x))


def _palabras(texto):
    return len(re.findall(r"\w+", str(texto)))


def revisa(n, valor, sustitucion, prediccion, datos, unidades):
    """Revisión de un ejercicio de la parte A.

    valor: lo que calculó la expresión del alumno. sustitucion: texto con sus números, el resultado y la
    unidad (p. ej. «70 / 2**8 = 0.273 °C»). prediccion: qué espera ver en la práctica, con sus palabras.
    datos: los números personales que deben aparecer en la sustitución. unidades: unidades aceptadas.
    """
    faltan = []
    if valor is None:
        faltan.append("la expresión en Python")
    elif not isinstance(valor, (int, float, np.integer, np.floating)) or not math.isfinite(float(valor)):
        faltan.append("una expresión que dé un número")
    s = str(sustitucion)
    if "=" not in s:
        faltan.append("la sustitución con «=»")
    # datos: lista de números (se nombran por su valor) o {descripción: valor} (se nombran sin revelar el valor)
    pares = datos.items() if isinstance(datos, dict) else [(None, d) for d in datos]
    for que, d in pares:
        if not aparece(d, s):
            faltan.append(f"{que} en la sustitución" if que else f"tu dato {float(d):g} en la sustitución")
    if valor is not None and not faltan and not aparece(valor, s.split("=")[-1]):
        faltan.append("el resultado que dio tu expresión, después del último «=»")
    if not any(u in s for u in unidades):
        faltan.append("la unidad (" + " o ".join(unidades) + ")")
    if _palabras(prediccion) < 10:
        faltan.append("lo que predice, en al menos 10 palabras")
    REGISTRO[n] = not faltan
    if faltan:
        raise ValueError(f"Ejercicio {n}: falta " + "; ".join(faltan) + ".")
    print(f"✔ Ejercicio {n} completo.")


def revisa_texto(nombre, texto, numeros, minimo=20):
    """Revisión de una respuesta escrita: largo mínimo y los números propios que debe citar.

    numeros: diccionario {qué es: valor}; el mensaje nombra lo que falta sin revelar el valor.
    """
    faltan = [que for que, x in numeros.items() if not aparece(x, texto)]
    if _palabras(texto) < minimo:
        faltan.insert(0, f"al menos {minimo} palabras")
    REGISTRO[nombre] = not faltan
    if faltan:
        raise ValueError(f"{nombre}: falta " + "; ".join(faltan) + ".")
    print(f"✔ {nombre} completo.")


import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 3.6), 'axes.spines.top': False, 'axes.spines.right': False})

def completa(valor, que):
    if valor is None:
        raise ValueError(f'Completa {que} (✎) y vuelve a ejecutar.')
    return valor

URL_DATOS = "https://raw.githubusercontent.com/ggt13/DAQ_TSU/main/sesion_04_error_e_incertidumbre/datos/intercomparacion_RC5.csv"
MODELOS = [  # registrador, exactitud del fabricante (± °C), resolución (°C)
    ("Elitech RC-5", 0.5, 0.1),
    ("Govee H5075", 0.3, 0.1),
    ("Elitech RC-4HC", 0.5, 0.1),
]
EXACT_RC5_C, RES_RC5_C = 0.5, 0.1
VENTANA_MIN = 60


def datos_s4(alumno):
    """Los datos personales de la parte A: un registrador, una referencia y sus lecturas."""
    modelo, exact, res = asignar(alumno, MODELOS, salto=1)
    t_ref = parametro(alumno, "T_ref", 20, 30, 2)
    sesgo = parametro(alumno, "sesgo", -0.8, 0.8, 2)
    ruido = parametro(alumno, "ruido", 0.05, 0.2, 3)
    n = asignar(alumno, [10, 12, 15, 20], salto=2)
    lect = np.round((t_ref + sesgo + alumno.rng("lecturas").normal(0, ruido, n)) / res) * res
    return dict(MODELO=modelo, EXACT_C=exact, RES_C=res, T_REF=t_ref, N_LECT=n, LECTURAS=np.round(lect, 1))


def incertidumbre(lecturas, exact, res):
    """u_A, u_B, u_res, u_c y U (k = 2) de una serie de lecturas."""
    u_a = np.std(lecturas, ddof=1) / np.sqrt(len(lecturas))
    u_b = exact / np.sqrt(3)
    u_r = res / np.sqrt(12)
    u_c = np.sqrt(u_a ** 2 + u_b ** 2 + u_r ** 2)
    return u_a, u_b, u_r, u_c, 2 * u_c


def clave_s4(alumno):
    d = datos_s4(alumno)
    u_a, u_b, u_r, u_c, U = incertidumbre(d["LECTURAS"], d["EXACT_C"], d["RES_C"])
    return {"u_A": u_a, "u_fab": u_b, "u_c": u_c, "U_exp": U, "delta_C": float(np.mean(d["LECTURAS"]) - d["T_REF"])}


def carga_intercomparacion(alumno):
    """La tabla de los tres RC-5 (minuto + tres columnas). Si no se puede leer, datos sintéticos marcados."""
    import pandas as pd
    try:
        tabla = pd.read_csv(URL_DATOS)
        fuente = "Intercomparación de los tres RC-5 del grupo"
    except Exception as e:
        print("⚠ No se pudo leer la intercomparación del grupo:", type(e).__name__, e)
        print("⚠ Se usan datos SINTÉTICOS. Avisa al profesor antes de entregar.")
        g = np.random.default_rng(2027)
        m = np.arange(0, 180, 1 / 6)
        sala = 24 + 0.6 * np.sin(m / 50) + 0.002 * m
        tabla = pd.DataFrame({"minuto": np.round(m, 3)})
        for k, sesgo in zip(("RC5_1", "RC5_2", "RC5_3"), (0.15, -0.25, 0.35)):
            tabla[k] = np.round(sala + sesgo + g.normal(0, 0.05, m.size), 1)
        fuente = "SINTÉTICOS (sin datos del grupo)"
    return tabla, fuente


def ventana_s4(alumno, tabla):
    """Tu registrador, tu ventana de 60 min y la referencia (mediana de los otros dos)."""
    loggers = list(tabla.columns[1:4])
    mio = asignar(alumno, loggers, salto=0)
    otros = [k for k in loggers if k != mio]
    dur = float(tabla["minuto"].iloc[-1])
    inicio = (int(alumno.codigo) % max(1, int(dur - VENTANA_MIN))) if dur > VENTANA_MIN else 0
    v = tabla[(tabla["minuto"] >= inicio) & (tabla["minuto"] < inicio + VENTANA_MIN)]
    return mio, otros, inicio, v["minuto"].to_numpy(), v[mio].to_numpy(), v[otros].median(axis=1).to_numpy()

print('Herramientas del curso listas.')

## Identificación
Tu nombre y tu código aparecen en cada gráfica y en el CSV. Sin ellos el cuaderno no avanza.

In [ ]:
NOMBRE = ""   # ✎ tu nombre completo
CODIGO = ""   # ✎ tu código de alumno
SESION = "S4"

alumno = identificar(NOMBRE, CODIGO, SESION)
print("Cuaderno de", alumno.nombre, "· código", alumno.codigo)

## Parte A · Actividad en clase (30 min)
Ejecuta la celda para ver **tu** registrador y sus lecturas. El termómetro de referencia marcó `T_REF` durante todas
las lecturas.

**Cómo se llena cada ejercicio.** Ejemplo con otros datos (no son los tuyos):
```
u_A    = np.std(LECTURAS, ddof=1) / np.sqrt(len(LECTURAS))
SUST_1 = "0.0800 °C / √16 = 0.0200 °C"
PRED_1 = "con más lecturas esta incertidumbre bajaría, pero sólo como la raíz del número de lecturas"
```

In [ ]:
globals().update(datos_s4(alumno))
print(f"Tu registrador · {MODELO} · exactitud ± {EXACT_C} °C · resolución {RES_C} °C")
print(f"Referencia: T_REF = {T_REF} °C · N_LECT = {N_LECT} lecturas:")
print("LECTURAS =", ", ".join(f"{x:.1f}" for x in LECTURAS))

### ★ Ejercicio 1 · Incertidumbre tipo A (3 pts)
**Para qué sirve:** mide cuánto cambian las lecturas por el ruido. Es la parte de la incertidumbre que se reduce
tomando más lecturas.

**u_A = s / √n**, donde *s* es la desviación estándar de las lecturas y *n* cuántas son. En Python:
`np.std(LECTURAS, ddof=1)` da *s* y `len(LECTURAS)` da *n*. En la sustitución escribe tu *s*, tu *n* y el resultado.

In [ ]:
# ✎ 1a · La expresión en Python, con las variables de tus datos
u_A = None   # ✎ escribe aquí
# ✎ 1b · La sustitución con TUS números, el resultado y la unidad
SUST_1 = ""   # ✎ escribe aquí
# ✎ 1c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_1 = ""   # ✎ escribe aquí

revisa(1, u_A, SUST_1, PRED_1, datos={'tu dato N_LECT': N_LECT, 'tu desviación estándar s': np.std(LECTURAS, ddof=1)}, unidades=['°C'])

### Ejercicio 2 · Incertidumbre tipo B del fabricante (2 pts)
**Para qué sirve:** la exactitud que declara el fabricante (± `EXACT_C`) es un error que no se ve en las lecturas y
no baja aunque tomes muchas.

Cuando sólo se sabe que el error está dentro de ± a, con cualquier valor igual de probable, la incertidumbre es la
semiamplitud entre la raíz de tres. Calcula `u_B` en °C.

In [ ]:
# ✎ 2a · La expresión en Python, con las variables de tus datos
u_B = None   # ✎ escribe aquí
# ✎ 2b · La sustitución con TUS números, el resultado y la unidad
SUST_2 = ""   # ✎ escribe aquí
# ✎ 2c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_2 = ""   # ✎ escribe aquí

revisa(2, u_B, SUST_2, PRED_2, datos={'tu dato EXACT_C': EXACT_C}, unidades=['°C'])

### Ejercicio 3 · Incertidumbre combinada (2 pts)
**Para qué sirve:** junta todo lo que puede equivocar la medición en un solo número.

Las fuentes independientes se suman **en cuadratura** (raíz de la suma de los cuadrados): tu `u_A`, tu `u_B` y la
de la resolución, `RES_C / np.sqrt(12)`. Calcula `u_c` en °C.

In [ ]:
# ✎ 3a · La expresión en Python, con las variables de tus datos
u_c = None   # ✎ escribe aquí
# ✎ 3b · La sustitución con TUS números, el resultado y la unidad
SUST_3 = ""   # ✎ escribe aquí
# ✎ 3c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_3 = ""   # ✎ escribe aquí

revisa(3, u_c, SUST_3, PRED_3, datos={'tu u_A': u_A, 'tu u_B': u_B}, unidades=['°C'])

### Ejercicio 4 · Incertidumbre expandida (2 pts)
**Para qué sirve:** es el número que se reporta: el intervalo valor ± U contiene al valor verdadero con alrededor
de 95 % de confianza.

Se obtiene multiplicando la combinada por el **factor de cobertura k = 2**. Calcula `U` en °C.

In [ ]:
# ✎ 4a · La expresión en Python, con las variables de tus datos
U = ""   # ✎ escribe aquí
# ✎ 4b · La sustitución con TUS números, el resultado y la unidad
SUST_4 = ""   # ✎ escribe aquí
# ✎ 4c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_4 = ""   # ✎ escribe aquí

revisa(4, U, SUST_4, PRED_4, datos={'tu u_c': u_c}, unidades=['°C'])

In [ ]:
media = float(np.mean(LECTURAS))
fig, ax = plt.subplots()
ax.plot(range(1, N_LECT + 1), LECTURAS, "o", color="#1F77B4", label="tus lecturas")
ax.axhspan(media - U, media + U, color="#1F77B4", alpha=0.12, label=f"media ± U = {media:.2f} ± {U:.2f} °C")
ax.axhline(T_REF, color="#C62828", ls="--", label=f"referencia {T_REF} °C")
ax.set_xlabel("lectura"); ax.set_ylabel("T (°C)"); ax.legend(loc="upper right", fontsize=8)
titulo(ax, f"{MODELO}: valor ± incertidumbre", alumno)
firma(fig, alumno); plt.show()

### Cierre · ¿Tu registrador es compatible? (1 pt)
Un registrador es **compatible** con la referencia si la diferencia entre su media y la referencia no pasa de su
`U`. Calcula `delta_C` (media de tus lecturas menos `T_REF`) y escribe tu veredicto con los dos números (al menos
20 palabras).

In [ ]:
delta_C = None   # ✎ escribe aquí
VEREDICTO = ""   # ✎ escribe aquí

completa(delta_C, 'la diferencia con la referencia')
revisa_texto('Veredicto', VEREDICTO, {'la diferencia con la referencia (en valor absoluto)': abs(delta_C),
             'tu incertidumbre expandida U': U})

## Parte B · Práctica (80 min)
**Objetivo:** aplicar lo mismo a un registro real y decidir si **tu** RC-5 es compatible con los otros dos.

Los tres RC-5 del grupo registraron juntos, en el mismo lugar. A cada alumno le toca uno y una ventana de 60 minutos.
Como no hay termómetro patrón, la **referencia** es la mediana de los otros dos en cada instante.

| Bloque | min | Qué haces |
|---|---|---|
| B1 · Datos | 10 | Cargas la intercomparación y ves los tres registros |
| B2 · Tu ventana | 15 | Separas tu registrador, tu hora y la referencia |
| B3 · Incertidumbre | 25 | Calculas tu u_A, tu U y tu diferencia con la referencia |
| B4 · Conclusión | 20 | Decides si tu registrador es compatible, con tus números |
| Tabla y captura | 10 | Descargas el CSV y lo importas en tu hoja de resultados |

In [ ]:
tabla, FUENTE = carga_intercomparacion(alumno)
fig, ax = plt.subplots()
for k in tabla.columns[1:4]:
    ax.plot(tabla["minuto"], tabla[k], lw=1, label=k)
ax.set_xlabel("minuto"); ax.set_ylabel("T (°C)"); ax.legend(ncol=3, fontsize=8)
titulo(ax, "Los tres RC-5 juntos", alumno)
firma(fig, alumno); plt.show()
print(f"B1 · {len(tabla)} lecturas por registrador · {tabla['minuto'].iloc[-1]:.0f} min · fuente: {FUENTE}")

In [ ]:
MIO, OTROS, INICIO, minuto, T_mio, T_ref = ventana_s4(alumno, tabla)
print(f"B2 · Tu registrador: {MIO} · referencia: mediana de {' y '.join(OTROS)}")
print(f"     Tu ventana: del minuto {INICIO} al {INICIO + VENTANA_MIN} · {len(T_mio)} lecturas (T_mio y T_ref)")

### B3 · Incertidumbre de tu registrador y diferencia con la referencia
Con las fórmulas de la parte A, ahora con `T_mio` y los datos del RC-5 (`EXACT_RC5_C`, `RES_RC5_C`):
1. `u_A_B`: la incertidumbre tipo A de `T_mio`.
2. `U_B`: la incertidumbre expandida (k = 2), combinando `u_A_B`, la del fabricante y la de la resolución.
3. `delta_B`: la media de `T_mio` menos la media de `T_ref`.

In [ ]:
u_A_B = None   # ✎ escribe aquí
U_B = ""   # ✎ escribe aquí
delta_B = None   # ✎ escribe aquí

for v, q in ((u_A_B, "u_A_B"), (U_B, "U_B"), (delta_B, "delta_B")):
    completa(v, q)
RESULTADOS = {"u_A": u_A, "u_fab": u_B, "u_c": u_c, "U_exp": U, "delta_C": float(delta_C),
              "u_A_rc5": float(u_A_B), "U_rc5": float(U_B), "delta_rc5": float(delta_B)}
fig, ax = plt.subplots()
ax.plot(minuto, T_ref, color="#888", lw=1.2, label="referencia (mediana de los otros dos)")
ax.fill_between(minuto, T_ref - U_B, T_ref + U_B, color="#888", alpha=0.15, label=f"referencia ± U = ± {U_B:.2f} °C")
ax.plot(minuto, T_mio, color="#D9822B", lw=1.2, label=f"{MIO}")
ax.set_xlabel("minuto"); ax.set_ylabel("T (°C)"); ax.legend(fontsize=8, loc="upper left")
titulo(ax, f"{MIO} contra la referencia", alumno)
firma(fig, alumno, RESULTADOS); plt.show()
print(f"u_A = {u_A_B:.4f} °C · U = {U_B:.3f} °C · diferencia = {delta_B:+.3f} °C → "
      + ("compatible" if abs(delta_B) <= U_B else "no compatible"))

### B4 · Conclusión
¿Tu RC-5 es compatible con la referencia? ¿Qué pesa más en su incertidumbre: el ruido (u_A) o el fabricante? Al
menos 25 palabras, con tu diferencia y tu U.

In [ ]:
CONCLUSION_B = ""   # ✎ escribe aquí

revisa_texto('Conclusión de la práctica', CONCLUSION_B,
             {'tu diferencia con la referencia': abs(delta_B), 'tu U': U_B}, minimo=25)

### Tabla y captura · CSV para tu hoja de resultados
La celda descarga `S4_<código>.csv`. En tu hoja de resultados: abre la pestaña **CSV** → **Archivo → Importar →
Subir** → elige el archivo → **Reemplazar la hoja actual**. La pestaña Resultados se llena sola.

In [ ]:
ruta = f"S4_{alumno.codigo}.csv"
exportar_csv(ruta, alumno, RESULTADOS,
             {"minuto": minuto, "T_mio_C": T_mio, "T_referencia_C": T_ref},
             extra={"fuente": FUENTE, "registrador": MIO, "ventana": f"minuto {INICIO} a {INICIO + VENTANA_MIN}"})
print("Archivo listo:", ruta, "· huella", huella(alumno, RESULTADOS))
try:
    from google.colab import files
    files.download(ruta)
except ImportError:
    pass

## Cierre del cuaderno
Antes de entregar: **Entorno de ejecución → Reiniciar y ejecutar todo**. Todo debe salir ✔.

In [ ]:
for k, ok in REGISTRO.items():
    print(("✔" if ok else "✘"), k if isinstance(k, str) else f"Ejercicio {k}")
print(f"\n{alumno.nombre} · {alumno.codigo} · {SESION} · {ahora()} · fuente: {FUENTE} · huella {huella(alumno, RESULTADOS)}")